# 05 — Report: сравнение конфигураций

Собирает метрики из ноутов 03-04 и формулирует сводный отчёт с полной
мета-информацией о прогоне: какие модели использовались, какие коллекции,
какие датасеты, параметры embedding/chunking.

In [ ]:
import sys
import json
from pathlib import Path
import glob as glob_mod
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")

from config import DATA_DIR

## 1. Загрузка данных

In [ ]:
print("Loading retrieval results...")
df = pd.read_csv(DATA_DIR / "retrieval_results.csv")
print(f"Loaded {len(df)} rows")
print(f"  Sources: {sorted(df['dataset_source'].unique())}")
print(f"  Collections: {sorted(df['collection'].unique())}")

In [ ]:
print("Loading generation traces...")
gen_traces = []
with open(DATA_DIR / "generation_traces.jsonl", encoding="utf-8") as f:
    for line in f:
        gen_traces.append(json.loads(line))
print(f"Loaded {len(gen_traces)} traces")
print(f"  Collections: {sorted({t['collection'] for t in gen_traces})}")

In [ ]:
print("Loading manifest...")
with open(DATA_DIR / "eval_manifest.json", encoding="utf-8") as f:
    manifest = json.load(f)
print(json.dumps(manifest, ensure_ascii=False, indent=2))

In [ ]:
print("Loading synthetic metadata...")
synth_meta_path = DATA_DIR / "synthetic_meta.json"
if synth_meta_path.exists():
    with open(synth_meta_path, encoding="utf-8") as f:
        synth_meta = json.load(f)
    print(json.dumps(synth_meta, ensure_ascii=False, indent=2))
else:
    print("  synthetic_meta.json not found")
    synth_meta = None

## 2. Run metadata

Полная информация о модели-судье, модели генерации, embedding, chunking.

In [ ]:
from config import LLM_MODEL, LLM_TEMPERATURE, LLM_MAX_TOKENS

print("=== Run Configuration ===")
print(f"LLM Judge: {LLM_MODEL}")
print(f"LLM Temperature: {LLM_TEMPERATURE}")
print(f"LLM Max Tokens: {LLM_MAX_TOKENS}")
if synth_meta:
    print(f"Synthetic Generator: {synth_meta.get('generator_model', 'N/A')}")
print(f"Embedding Model: {manifest['embedding']['model_name']}")
print(f"Embedding Vector Size: {manifest['embedding']['vector_size']}")
print(f"Query Prefix: {manifest['embedding']['query_prefix']}")
print(f"Document Prefix: {manifest['embedding']['document_prefix']}")
print(
    f"Chunking: size={manifest['chunking']['chunk_size']}, "
    f"overlap={manifest['chunking']['chunk_overlap']}, "
    f"tokenizer={manifest['chunking']['tokenizer']}"
)
print(f"Collections: {manifest['collections']}")
print(f"Datasets: {manifest['datasets']}")
print(
    f"Retrieval: top_k={manifest.get('top_k', 5)}, "
    f"k_set={manifest.get('k_set', (1, 3, 5))}, "
    f"threshold={manifest.get('relevance_threshold', 0.3)}"
)

## 3. Retrieval aggregation

In [ ]:
pivot = df.groupby(["dataset_source", "collection"])[
    ["recall@1", "recall@3", "recall@5", "precision@5", "mrr", "ndcg@5"]
].mean()
print(f"Summary: {len(pivot)} rows")
pivot.round(3)

## 4. Generation results summary

In [ ]:
gen_df = pd.DataFrame(gen_traces)
print(f"Generation traces: {len(gen_df)} rows")

# Count answers vs declines per collection
gen_summary = (
    gen_df.groupby("collection")
    .agg(
        total_queries=("answer", "count"),
        answered=("answer", lambda x: sum(1 for v in x if v and v.strip())),
        with_context=("retrieved_chunks", lambda x: sum(1 for v in x if v)),
    )
    .reset_index()
)
gen_summary["answer_rate"] = gen_summary["answered"] / gen_summary["total_queries"]
gen_summary["context_rate"] = gen_summary["with_context"] / gen_summary["total_queries"]
print(gen_summary.to_string(index=False))

## 4a. DeepEval generation metrics

In [ ]:
print("Loading generation evaluation results...")
pattern = str(DATA_DIR / "generation_results_deepeval_*.csv")
csv_files = sorted(glob_mod.glob(pattern))
print(f"Found {len(csv_files)} result file(s)")

if csv_files:
    valid_metrics = [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
        "answer_correctness",
    ]
    all_rows = []
    for csv_path in csv_files:
        col_df = pd.read_csv(csv_path)
        print(f"  {Path(csv_path).name}: {len(col_df)} rows")
        all_rows.append(col_df)

    gen_results = pd.concat(all_rows, ignore_index=True)
    score_cols = [c for c in valid_metrics if c in gen_results.columns]
    if score_cols:
        pivot = gen_results.groupby("collection")[score_cols].mean()
        print("\nMean metrics per collection:")
        print(pivot.round(3).to_string())
else:
    print("No generation result files found. Run 04 notebook first.")
    gen_results = pd.DataFrame()

## 5. Визуализация

In [ ]:
print("Generating recall@5 chart...")
sources = sorted(df["dataset_source"].unique())
fig, axes = plt.subplots(1, len(sources), figsize=(6 * len(sources), 5), sharey=True)
if len(sources) == 1:
    axes = [axes]
for ax, source in zip(axes, sources):
    sub = df[df["dataset_source"] == source].groupby("collection")["recall@5"].mean()
    ax.bar(sub.index, sub.values)
    ax.set_title(source)
    ax.set_xlabel("collection")
    ax.set_ylabel("recall@5")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
out = DATA_DIR / "recall_by_collection.png"
plt.savefig(out, dpi=120)
print(f"Saved -> {out}")
plt.show()

gen_results = locals().get("gen_results")
if gen_results is not None and not gen_results.empty:
    valid_metrics = [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
        "answer_correctness",
    ]
    score_cols = [c for c in valid_metrics if c in gen_results.columns]
    if score_cols:
        print("\nGenerating generation metrics chart...")
        pivot_gen = gen_results.groupby("collection")[score_cols].mean()
        fig2, ax2 = plt.subplots(figsize=(8, 5))
        pivot_gen.plot(kind="bar", ax=ax2)
        ax2.set_title("Generation metrics per collection")
        ax2.set_xlabel("collection")
        ax2.set_ylabel("score")
        ax2.legend(loc="best")
        ax2.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        out2 = DATA_DIR / "generation_metrics_by_collection.png"
        plt.savefig(out2, dpi=120)
        print(f"Saved -> {out2}")
        plt.show()